# nb08 — WakeHuBERT Inference (CPU-friendly)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TigreGotico/ww-trainer/blob/dev/notebooks/nb08_wakehubert.ipynb)

> **Build a wake-word detector on top of a distilled HuBERT featurizer — no GPU needed at this stage.**  \
> A *featurizer* turns raw audio into numbers a model can learn from. Instead of the\
> hand-designed MFCC features used in the quickstart, this notebook uses a small neural\
> *speech* featurizer (`tinyhubert.onnx`) that already understands phonemes, then trains a\
> tiny classifier on top of it.

---

## Background — the jargon, defined

- **Featurizer** — the front-end that converts a raw audio waveform into a compact numeric
  representation (a *feature vector* per short time frame). Everything downstream learns
  from these features rather than from raw samples.
- **MFCC** (Mel-Frequency Cepstral Coefficients) — the classic *hand-designed* speech
  featurizer. Cheap and tiny, but it knows nothing about language — it is pure signal
  processing.
- **HuBERT** — a large neural network from Meta, *self-supervised* on 960 hours of speech.
  It learns phoneme-like representations, so its features carry far more linguistic
  information than MFCC. Full HuBERT is ~350 MB — too heavy for an always-on device.
- **Distillation** — training a small "student" network to imitate a large "teacher".
  Notebook nb07 distils HuBERT down to `tinyhubert.onnx` (~5–15 MB). This notebook
  *uses* that student; it does not create it.
- **ONNX** — an open, framework-neutral model format. An ONNX model runs via
  `onnxruntime` with no PyTorch needed at deployment — ideal for embedded devices.
- **GRU / FFN** — the two classifier-head options. A **GRU** (Gated Recurrent Unit) reads
  the feature frames in sequence and remembers context over time; an **FFN**
  (Feed-Forward Network) is a simpler, cheaper fully-connected head with no memory.

---

## The two-step pipeline

```
Step 1 (nb07, needs a GPU): distil HuBERT → tinyhubert.onnx  (~5–15 MB, causal CNN+GRU)
Step 2 (this notebook, CPU only): tinyhubert.onnx + GRU/FFN head → wake-word classifier
```

This notebook is **Step 2**. It assumes you already produced `tinyhubert.onnx`. If you
have not, run `nb07_distill.ipynb` first (it needs a GPU); the file path is configured in
the first code cell below.

**Why 768-dimensional embeddings help.** HuBERT's features are 768 numbers per frame and
encode phoneme-level structure. Because that structure is already extracted, the
classifier on top only has to learn a shallow mapping ("does this sequence of phonemes
sound like the wake word?"). With raw MFCC the classifier must learn far more from scratch.

---

## What this notebook does

| Cell | Step | What happens |
|------|------|-------------|
| 1 | **Configure** | Set wake phrase, featurizer path, classifier type |
| 2 | **Install** | Install `ww_trainer` and audio dependencies |
| 3 | **Dataset** | Reuse or generate the labelled train/test audio |
| 4 | **Check featurizer** | Load `tinyhubert.onnx`, run one forward pass, confirm shape |
| 5 | **Train** | Train the GRU/FFN head on top of the frozen featurizer |
| 6 | **Compare** | WakeHuBERT vs the MFCC baseline (F1 bar chart) |
| 7 | **Visualise** | PCA scatter of embeddings — are wake/non-wake separable? |
| 8 | **Size audit** | Report deployed ONNX size (featurizer + head) |
| 9 | **Inference test** | Score a real positive sample end-to-end |

This notebook is **resume-safe**: with `SKIP_COMPLETED=true` it reloads a cached result
instead of retraining, so you can re-run cells freely.

---

## Size audit (what you ship)

| Component | Size | Notes |
|-----------|------|-------|
| `tinyhubert.onnx` (featurizer) | ~5–15 MB | the dominant cost |
| Classifier head ONNX | < 500 KB | tiny GRU/FFN |
| **Total deployed** | **~6–16 MB** | both files together |

For comparison: a full (undistilled) HuBERT featurizer is ~350 MB, and a plain MFCC
featurizer is only ~5–20 KB. WakeHuBERT trades a larger model for richer features — useful
when accuracy matters more than the last few megabytes. For microcontroller targets, use
the MFCC tiers in the quickstart / nb04 instead.

## Cell 1 — Configuration

**The main cell to edit.** Set your wake phrase and point `FEATURIZER_ONNX` at the
`tinyhubert.onnx` produced by nb07.

Every setting can also be supplied as an environment variable of the same name —
`os.environ.get(...)` picks those up, so you can drive the notebook from outside without
editing code.

Key settings:

- `WAKE_WORD` — the phrase to detect.
- `FEATURIZER_ONNX` — path to the distilled featurizer. Defaults to
  `OUTPUT_DIR/tinyhubert.onnx`.
- `FEATURE_DIM` — embedding width the featurizer outputs (768 for HuBERT). Cell 4
  auto-corrects this if it does not match the actual model.
- `ARCH` — classifier head: `gru` (sequence-aware, more accurate) or `ffn` (smaller).
- `SKIP_COMPLETED` — when `true`, reuse a cached training result instead of retraining.

> Leave `MLFLOW_*` blank unless you run an MLflow experiment-tracking server — it is
> entirely optional and has no effect on the trained model.

In [ ]:
import os

# ── Core ──────────────────────────────────────────────────────────────────────
WAKE_WORD          = os.environ.get("WAKE_WORD",          "hey jarvis")
OUTPUT_DIR         = os.environ.get("OUTPUT_DIR",         "./ww_output")
DEVICE             = os.environ.get("DEVICE",             "auto")
SEED               = int(os.environ.get("SEED",           "42"))

# ── Featurizer ────────────────────────────────────────────────────────────────
# Path to tinyhubert.onnx produced by nb07
FEATURIZER_ONNX    = os.environ.get("FEATURIZER_ONNX",    f"{OUTPUT_DIR}/tinyhubert.onnx")
FEATURE_DIM        = int(os.environ.get("FEATURE_DIM",    "768"))

# ── Classifier ────────────────────────────────────────────────────────────────
ARCH               = os.environ.get("ARCH",               "gru")  # gru or ffn
EPOCHS             = int(os.environ.get("EPOCHS",         "50"))
BATCH_SIZE         = int(os.environ.get("BATCH_SIZE",     "16"))
SKIP_COMPLETED     = os.environ.get("SKIP_COMPLETED",     "true").lower() == "true"

# ── Dataset ───────────────────────────────────────────────────────────────────
N_POSITIVE         = int(os.environ.get("N_POSITIVE",     "500"))
LANG               = os.environ.get("LANG",               "en")
ADVERSARIAL        = os.environ.get("ADVERSARIAL",        "true").lower() == "true"
DOWNLOAD_AUGMENT   = os.environ.get("DOWNLOAD_AUGMENT",   "true").lower() == "true"
REUSE_DATASET      = os.environ.get("REUSE_DATASET",      "true").lower() == "true"
CUSTOM_TRAIN_CSV   = os.environ.get("CUSTOM_TRAIN_CSV",   "")
CUSTOM_TEST_CSV    = os.environ.get("CUSTOM_TEST_CSV",    "")

# ── MLflow (optional) ────────────────────────────────────────────────────────
MLFLOW_URI         = os.environ.get("MLFLOW_URI",         "")
MLFLOW_SECRET      = os.environ.get("MLFLOW_SECRET",      "MLFLOW_TOKEN")

import pathlib
pathlib.Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f"Wake word      : {WAKE_WORD!r}")
print(f"Featurizer ONNX: {FEATURIZER_ONNX}")
print(f"Feature dim    : {FEATURE_DIM}")
print(f"Classifier arch: {ARCH!r}  |  Epochs: {EPOCHS}")

## Cell 2 — Install dependencies and detect platform

Installs `ww_trainer` plus the audio/ML stack and the OVOS plugins used for dataset
generation (TTS for synthesising the wake phrase, a voice-activity detector for trimming
silence). Already-installed packages are skipped, so this is safe to re-run.

It also detects whether you are on Kaggle, Paperspace, Colab, or a local machine, caps the
CPU thread count for predictable timing, and — on Kaggle — injects an optional MLflow token
from Kaggle Secrets.

> **No GPU is required.** The featurizer runs through ONNX Runtime on CPU; the small head
> trains quickly on CPU too. A GPU, if present, is used automatically.

In [ ]:
import subprocess, sys, os

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

_pip("torch", "torchaudio", "soundfile", "numpy", "scikit-learn",
     "matplotlib", "pandas", "librosa", "onnx", "onnxruntime", "click", "tqdm")
_pip("ovos-plugin-manager", "ovos-tts-plugin-edge-tts",
     "git+https://github.com/TigreGotico/vadonnx.git", "datasets")
try:
    import ww_trainer
except ImportError:
    _pip("ww_trainer")

_platform = (
    "kaggle"     if os.path.exists("/kaggle")     else
    "paperspace" if os.path.exists("/notebooks")  else
    "colab"      if "google.colab" in sys.modules else
    "local"
)

import torch
torch.set_num_threads(min(12, os.cpu_count() or 4))
os.environ.setdefault("OMP_NUM_THREADS", str(min(12, os.cpu_count() or 4)))

if _platform == "kaggle" and MLFLOW_SECRET:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret(MLFLOW_SECRET)
        os.environ["MLFLOW_TRACKING_TOKEN"] = token
    except Exception as e:
        print(f"MLflow secret not found: {e}")
if MLFLOW_URI:
    os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_URI

print(f"Platform : {_platform} | CUDA: {torch.cuda.is_available()} | threads: {torch.get_num_threads()}")

## Cell 3 — Prepare the dataset

Builds (or reuses) the labelled audio the classifier learns from:

- **Positives** — utterances of your wake phrase, synthesised with text-to-speech in many
  voices, labelled `1`.
- **Negatives** — speech that does *not* contain the phrase, labelled `0`.
- **Adversarial negatives** (when `ADVERSARIAL=true`) — phonetically-similar near-misses,
  which make the model harder to fool.
- **Augmentation audio** (when `DOWNLOAD_AUGMENT=true`) — background noise, music, and room
  reverb mixed in during training to mimic real rooms.

Three ways to get data, in priority order:

1. **`CUSTOM_TRAIN_CSV`** set → use your own CSV (auto-split 80/20 into train/test if no
   `CUSTOM_TEST_CSV` is given).
2. **`REUSE_DATASET=true`** and a dataset already exists → reuse it instantly (no
   downloads).
3. Otherwise → generate a fresh dataset (this is the slow path: TTS + downloads).

The augmentation folders, when present, are collected into `_aug_kwargs_full` and passed to
training in Cell 5.

> A *CSV manifest* is just a two-column file: `path_to_audio.wav,label` where label is `1`
> or `0`.

In [ ]:
import shutil
from pathlib import Path

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(OUTPUT_DIR).free / 1e9
assert free_gb > 3, f"Only {free_gb:.1f} GB free — need at least 3 GB."
print(f"Disk free: {free_gb:.1f} GB")

_aug_kwargs_full = {}

if CUSTOM_TRAIN_CSV:
    import random, csv as _csv
    from ww_trainer.utils import read_dataset_csv
    train_csv = Path(CUSTOM_TRAIN_CSV)
    test_csv  = Path(CUSTOM_TEST_CSV) if CUSTOM_TEST_CSV else None
    if test_csv is None:
        split_dir = Path(OUTPUT_DIR) / "dataset_split"
        split_dir.mkdir(parents=True, exist_ok=True)
        split_train = split_dir / "train.csv"
        split_test  = split_dir / "test.csv"
        if not split_train.exists():
            rows = read_dataset_csv(train_csv)
            random.seed(SEED); random.shuffle(rows)
            cut = int(len(rows) * 0.8)
            for p, rs in [(split_train, rows[:cut]), (split_test, rows[cut:])]:
                with open(p, "w", newline="") as f:
                    _csv.writer(f).writerows(rs)
        train_csv, test_csv = split_train, split_test
else:
    from ww_trainer.datagen import DatagenConfig, run_datagen_pipeline, DatagenResult, normalize_wake_word
    dataset_dir = Path(OUTPUT_DIR) / "dataset"
    _train_csv_check = dataset_dir / "train" / "metadata.csv"
    if REUSE_DATASET and _train_csv_check.exists():
        slug = normalize_wake_word(WAKE_WORD)
        _dr = DatagenResult(
            train_csv=dataset_dir / "train" / "metadata.csv",
            test_csv=dataset_dir / "test" / "metadata.csv",
            positives_dir=dataset_dir / slug / "positives",
            negatives_dir=dataset_dir / slug / "negatives",
            bg_noise_dir=dataset_dir / "augmentation" / "bg_noise",
            music_dir=dataset_dir / "augmentation" / "music",
            rir_dir=dataset_dir / "augmentation" / "rir",
        )
        print(f"Reusing dataset at {dataset_dir}")
    else:
        _dr = run_datagen_pipeline(DatagenConfig(
            wake_word=WAKE_WORD, output_dir=dataset_dir,
            n_positive=N_POSITIVE, lang=LANG,
            adversarial=ADVERSARIAL, vad_trim=True,
            download_augmentation=DOWNLOAD_AUGMENT, seed=SEED,
        ))
    train_csv, test_csv = _dr.train_csv, _dr.test_csv
    for attr, key in [("bg_noise_dir", "bg_noise_folder"),
                      ("music_dir", "music_folder"),
                      ("rir_dir", "rir_folder")]:
        d = getattr(_dr, attr, None)
        if d and Path(d).exists():
            _aug_kwargs_full[key] = str(d)

print(f"train_csv: {train_csv}")
print(f"test_csv : {test_csv}")

## Cell 4 — Check the featurizer

Before training, this cell loads `tinyhubert.onnx` through `OnnxFeatureExtractor`, feeds it
one second of dummy audio, and prints the output shape. This is a fast sanity check that
the featurizer file exists and produces the expected `(frames, FEATURE_DIM)` tensor.

`OnnxFeatureExtractor` is the runtime wrapper around the ONNX featurizer:

```python
from ww_trainer.feats import OnnxFeatureExtractor
extractor = OnnxFeatureExtractor("tinyhubert.onnx")
features = extractor(waveform)   # input: (T,) float32 @ 16 kHz → output: (T', D)
```

HuBERT-style models use a 320-sample hop at 16 kHz, so they emit roughly
`16000 / 320 ≈ 50` feature frames per second. If the model's real embedding width differs
from `FEATURE_DIM`, the cell prints a warning and **auto-corrects** `FEATURE_DIM` so the
rest of the notebook stays consistent. If the file is missing, it raises a clear error
pointing you back to nb07.

In [ ]:
import numpy as np
import onnxruntime as ort
from pathlib import Path

# ── Check featurizer ─────────────────────────────────────────────────────────
# Load OnnxFeatureExtractor, run one forward pass, print output shape.

feat_path = Path(FEATURIZER_ONNX)
if not feat_path.exists():
    raise FileNotFoundError(
        f"Featurizer ONNX not found: {feat_path}\n"
        f"Run nb07_distill.ipynb first to generate tinyhubert.onnx."
    )

print(f"Featurizer: {feat_path}  ({feat_path.stat().st_size / 1024 / 1024:.1f} MB)")

try:
    from ww_trainer.feats import OnnxFeatureExtractor
    _extractor = OnnxFeatureExtractor(str(feat_path))
    dummy_wav = np.random.randn(16000).astype(np.float32)
    feat_out = _extractor(dummy_wav)
    print(f"OnnxFeatureExtractor output shape: {feat_out.shape}")
    print(f"  Expected: (T', {FEATURE_DIM}) — T' ≈ {16000//320} frames per second")
    _feature_dim_actual = feat_out.shape[-1]
    if _feature_dim_actual != FEATURE_DIM:
        print(f"  WARNING: actual dim={_feature_dim_actual} != FEATURE_DIM={FEATURE_DIM}")
        print(f"  Set FEATURE_DIM={_feature_dim_actual} to match.")
        FEATURE_DIM = _feature_dim_actual
except (ImportError, AttributeError):
    print("OnnxFeatureExtractor not in ww_trainer.feats — using raw onnxruntime")
    sess = ort.InferenceSession(str(feat_path), providers=["CPUExecutionProvider"])
    in_name = sess.get_inputs()[0].name
    dummy_np = np.random.randn(1, 16000).astype(np.float32)
    out = sess.run(None, {in_name: dummy_np})
    print(f"onnxruntime output shape: {out[0].shape}")
    _feature_dim_actual = out[0].shape[-1]
    FEATURE_DIM = _feature_dim_actual

print(f"Using FEATURE_DIM={FEATURE_DIM}")

## Cell 5 — Train the classifier head

This trains the wake-word classifier *on top of* the frozen featurizer. The featurizer
weights are not touched — only the small GRU/FFN head learns. That is why this step is
CPU-friendly and fast.

The call uses `tier="onnx_{ARCH}"` (e.g. `onnx_gru`) to tell the trainer to take features
from an external ONNX file (`featurizer_onnx=FEATURIZER_ONNX`) instead of computing MFCCs.
The augmentation folders collected in Cell 3 are forwarded via `**_aug_kwargs_full`.

**Result caching.** With `SKIP_COMPLETED=true`, a previously saved
`wakehubert_result.json` is reloaded instead of retraining — handy across sessions. The
result records F1, precision, recall, runtime, and the path to the exported head ONNX.

> **F1 score** is the harmonic mean of *precision* (of the audio flagged as the wake word,
> how much really was) and *recall* (of all real wake-word audio, how much was caught).
> F1 sits in `[0, 1]`; higher is better. It is a single number that balances false alarms
> against missed detections.

> **Troubleshooting:** if you see `tier 'onnx_gru' not found`, your installed `ww_trainer`
> version may not register ONNX-featurizer tiers. The cell catches the error, records it in
> the result JSON, and prints a hint — the rest of the notebook still runs against whatever
> result is available.

In [ ]:
import json, time
from pathlib import Path
from ww_trainer.quickstart import train_from_wakeword

# ── Train: WakeHuBERT featurizer + GRU/FFN head ───────────────────────────────
# Uses featurizer_type="onnx" to tell the trainer to use an external ONNX featurizer.

model_subdir  = Path(OUTPUT_DIR) / "models" / "wakehubert"
result_file   = Path(OUTPUT_DIR) / "wakehubert_result.json"

if SKIP_COMPLETED and result_file.exists():
    print(f"Loading cached result from {result_file}")
    _wh_result = json.loads(result_file.read_text())
    print(f"  F1={_wh_result.get('f1', 0):.4f}  status={_wh_result.get('status')}")
else:
    t0 = time.time()
    try:
        r = train_from_wakeword(
            WAKE_WORD,
            str(model_subdir),
            tier=f"onnx_{ARCH}",        # onnx_gru or onnx_ffn
            featurizer_onnx=FEATURIZER_ONNX,
            feature_dim=FEATURE_DIM,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            device=DEVICE,
            seed=SEED,
            reuse_dataset=True,
            **_aug_kwargs_full,
        )
        elapsed = time.time() - t0
        _wh_result = {
            "tier": f"wakehubert_{ARCH}",
            "f1": r.metrics.get("f1", 0.0),
            "precision": r.metrics.get("precision", 0.0),
            "recall": r.metrics.get("recall", 0.0),
            "elapsed_s": round(elapsed, 1),
            "head_onnx": str(r.best_onnx_path) if r.best_onnx_path else "",
            "feat_onnx": str(feat_path),
            "status": "ok",
        }
        print(f"DONE: F1={_wh_result['f1']:.4f}  ({elapsed:.0f}s)")
    except Exception as exc:
        elapsed = time.time() - t0
        _wh_result = {
            "tier": f"wakehubert_{ARCH}",
            "f1": 0.0, "precision": 0.0, "recall": 0.0,
            "elapsed_s": round(elapsed, 1), "head_onnx": "",
            "feat_onnx": str(feat_path),
            "status": f"error: {exc}",
        }
        print(f"ERROR: {exc}")
        print()
        print("Hint: if the tier 'onnx_gru' is not found, check ww_trainer.tiers for ONNX-featurizer tiers.")
        print("You may need to train the GRU head manually with featurizer_type='onnx'.")
    result_file.write_text(json.dumps(_wh_result, indent=2))

## Cell 6 — Compare against the MFCC baseline

The whole point of WakeHuBERT is richer features — so the natural question is *"is it
actually better than plain MFCC?"* This cell loads an MFCC `small`-tier result from an
earlier notebook (nb05/nb01, if you ran one) and plots the two F1 scores side by side.

**How to read the chart:** each bar is one model's F1 on the same test set. A taller blue
(WakeHuBERT) bar than the coral (MFCC) bar means the neural featurizer earned its extra
megabytes for *this* wake word. Sometimes MFCC is competitive — wake words with very
distinctive sounds are easy, and the heavier featurizer buys little. The honest answer is
always dataset-specific, which is exactly why the comparison is run.

If no MFCC baseline file is found, the cell says so and just shows the WakeHuBERT result —
run nb05 first if you want the comparison.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ── Compare: WakeHuBERT vs MFCC-40 baseline ───────────────────────────────────
# Load MFCC results from nb05 (small tier) or nb01 if available.

compare_rows = [_wh_result]

# Search for MFCC baseline results
_mfcc_search_paths = [
    Path(OUTPUT_DIR) / "embedded_results" / "small.json",
    Path(OUTPUT_DIR) / "micro_results" / "small.json",
    Path(OUTPUT_DIR) / "gpu_results" / "small.json",
]
for p in _mfcc_search_paths:
    if p.exists():
        r = json.loads(p.read_text())
        r.setdefault("tier", "small_mfcc")
        compare_rows.append(r)
        print(f"Loaded MFCC baseline from {p}: F1={r.get('f1', 0):.4f}")
        break
else:
    print("No MFCC baseline found — run nb05_embedded.ipynb first for comparison.")

df_cmp = pd.DataFrame(compare_rows)
_cols = ["tier", "f1", "precision", "recall", "elapsed_s", "status"]
_cols = [c for c in _cols if c in df_cmp.columns]
print()
print("WakeHuBERT vs MFCC baseline:")
print(df_cmp[_cols].to_string(index=False))

if len(df_cmp) > 1:
    df_ok = df_cmp[df_cmp["status"] == "ok"].copy()
    if not df_ok.empty:
        fig, ax = plt.subplots(figsize=(7, 4))
        colors = ["steelblue" if "wakehubert" in str(t) else "coral"
                  for t in df_ok["tier"]]
        bars = ax.bar(df_ok["tier"], df_ok["f1"], color=colors, edgecolor="white")
        ax.set_ylabel("F1")
        ax.set_title(f"WakeHuBERT vs MFCC — {WAKE_WORD!r}")
        ax.set_ylim(0, 1.05)
        for bar, f1 in zip(bars, df_ok["f1"]):
            ax.text(bar.get_x() + bar.get_width()/2, f1 + 0.01, f"{f1:.3f}",
                    ha="center", va="bottom", fontsize=9)
        plt.tight_layout()
        cmp_path = str(Path(OUTPUT_DIR) / "wakehubert_comparison.png")
        plt.savefig(cmp_path, dpi=120, bbox_inches="tight")
        plt.show()
        print(f"Comparison saved: {cmp_path}")

## Cell 7 — Visualise the embeddings (PCA)

This cell offers an intuition for *why* one featurizer beats another. It takes up to 200
test clips, runs each through the featurizer, mean-pools the frames into a single vector per
clip, then projects those high-dimensional vectors down to 2-D with **PCA** (Principal
Component Analysis — a standard way to flatten many dimensions onto the two axes that
capture the most variation).

**How to read the scatter plot:** each dot is one audio clip — blue for the wake word,
coral for non-wake. If the two colours form **separate clusters**, the featurizer already
makes the classes easy to tell apart, and even a tiny classifier head will do well. If the
colours are **mixed together**, the featurizer is not separating the classes and accuracy
will suffer. The left panel shows WakeHuBERT; the right shows MFCC (when an nb05 featurizer
is available) for direct comparison.

The "variance explained" figure in each title is how much of the original spread these two
axes capture — higher means the 2-D picture is more faithful to the real high-dimensional
geometry.

In [ ]:
import numpy as np
import csv
import torchaudio
import onnxruntime as ort
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from pathlib import Path

# ── Embedding visualisation: PCA of WakeHuBERT embeddings ─────────────────────
# Compare to MFCC-40 PCA if available (from nb05).

sess = ort.InferenceSession(str(feat_path), providers=["CPUExecutionProvider"])
in_name = sess.get_inputs()[0].name

wavs_pca, labels_pca = [], []
with open(test_csv) as f:
    for row in csv.reader(f):
        if len(row) < 2 or not Path(row[0]).exists():
            continue
        wav, sr = torchaudio.load(row[0])
        if sr != 16000:
            wav = torchaudio.functional.resample(wav, sr, 16000)
        wavs_pca.append(wav.mean(0).numpy().astype(np.float32))
        labels_pca.append(int(row[1].strip()))
        if len(wavs_pca) >= 200:
            break

labels_arr = np.array(labels_pca)

# Extract WakeHuBERT embeddings
wh_embs = []
for wav in wavs_pca:
    out = sess.run(None, {in_name: wav[np.newaxis, :]})[0]
    wh_embs.append(out.mean(axis=1).ravel() if out.ndim == 3 else out.ravel())
wh_embs = np.array(wh_embs)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(f"Embedding PCA — {WAKE_WORD!r}", fontsize=12)

# WakeHuBERT PCA
pca_wh = PCA(n_components=2, random_state=SEED)
coords_wh = pca_wh.fit_transform(wh_embs)
for lbl, color, name in [(1, "steelblue", "wake"), (0, "coral", "non-wake")]:
    mask = labels_arr == lbl
    axes[0].scatter(coords_wh[mask, 0], coords_wh[mask, 1],
                    c=color, s=20, alpha=0.6, label=name)
axes[0].set_title(f"WakeHuBERT (768-d)\nvar explained: {pca_wh.explained_variance_ratio_.sum():.2f}")
axes[0].legend(fontsize=8)

# MFCC PCA from nb05 (if available)
_mfcc_feat_onnx = None
for search_dir in [
    Path(OUTPUT_DIR) / "models" / "small" / "model",
    Path(OUTPUT_DIR) / "models" / "small",
]:
    candidate = search_dir / "best_f1_featurizer.onnx"
    if candidate.exists():
        _mfcc_feat_onnx = str(candidate)
        break

if _mfcc_feat_onnx:
    sess_mfcc = ort.InferenceSession(_mfcc_feat_onnx, providers=["CPUExecutionProvider"])
    in_mfcc = sess_mfcc.get_inputs()[0].name
    mfcc_embs = []
    for wav in wavs_pca:
        out = sess_mfcc.run(None, {in_mfcc: wav[np.newaxis, :]})[0]
        mfcc_embs.append(out.ravel() if out.ndim <= 2 else out.mean(axis=1).ravel())
    mfcc_embs = np.array(mfcc_embs)
    pca_mfcc = PCA(n_components=2, random_state=SEED)
    coords_mfcc = pca_mfcc.fit_transform(mfcc_embs)
    for lbl, color, name in [(1, "steelblue", "wake"), (0, "coral", "non-wake")]:
        mask = labels_arr == lbl
        axes[1].scatter(coords_mfcc[mask, 0], coords_mfcc[mask, 1],
                        c=color, s=20, alpha=0.6, label=name)
    axes[1].set_title(f"MFCC-40 (small tier)\nvar explained: {pca_mfcc.explained_variance_ratio_.sum():.2f}")
    axes[1].legend(fontsize=8)
else:
    axes[1].text(0.5, 0.5, "MFCC baseline not available\nRun nb05 first",
                 ha="center", va="center", transform=axes[1].transAxes, fontsize=10)
    axes[1].set_title("MFCC-40 (not available)")

plt.tight_layout()
pca_path = str(Path(OUTPUT_DIR) / "wakehubert_pca.png")
plt.savefig(pca_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"PCA plot saved: {pca_path}")

## Cell 8 — Size audit

Reports the on-disk size of everything you would deploy: the featurizer
(`tinyhubert.onnx`) plus the trained classifier head. The featurizer dominates — the head
is well under a megabyte — so the total is essentially the featurizer size.

The printed reference line puts this in context against a full HuBERT featurizer (~350 MB)
and a plain MFCC featurizer (~5–20 KB). The takeaway: WakeHuBERT is a middle ground. If
your target is a microcontroller where every kilobyte counts, the MFCC tiers (quickstart /
nb04) remain the right choice.

In [ ]:
from pathlib import Path

# ── ONNX export + size audit ──────────────────────────────────────────────────
# The featurizer (tinyhubert.onnx) is pre-existing.
# The classifier head is exported during train_from_wakeword().

print("Size audit:")
print()

feat_p = Path(FEATURIZER_ONNX)
head_p = Path(_wh_result.get("head_onnx", "")) if _wh_result.get("head_onnx") else None

feat_mb = feat_p.stat().st_size / 1024 / 1024 if feat_p.exists() else None
head_kb = head_p.stat().st_size / 1024 if (head_p and head_p.exists()) else None
total_mb = (feat_mb or 0) + ((head_kb or 0) / 1024)

print(f"  Featurizer (tinyhubert.onnx): {feat_mb:.1f} MB" if feat_mb else "  Featurizer: MISSING")
print(f"  Classifier head ONNX       : {head_kb:.0f} KB" if head_kb else "  Classifier head: MISSING")
print(f"  Total deployed size        : {total_mb:.1f} MB")
print()
print("Reference:")
print("  Full HuBERT featurizer ONNX : ~350 MB")
print("  MFCC-40 featurizer ONNX     : ~5–20 KB")
print("  TinyHuBERT (this notebook)  : ~5–15 MB")
print()
print("Note: tinyhubert.onnx is the bottleneck.")
print("For MCU deployment, use nb04 (MFCC tiers) instead.")

## Cell 9 — End-to-end inference test

The final check chains both ONNX files exactly as a device would: raw audio →
`featurizer.onnx` → features → `head.onnx` → confidence score. It uses
`OnnxWakeWordInferencer` (the same class OVOS uses at runtime, `onnxruntime` only, no
PyTorch) to score one real positive sample from the test set.

A `score > 0.5` prints `PASS` — the model recognises its own wake word. A low score points
to undertraining: increase `EPOCHS`, add more positives (`N_POSITIVE`), or rebuild the
dataset with `REUSE_DATASET=false`.

The cell ends with a ready-to-copy CLI command for testing the model on any WAV file:

```bash
python scripts/eval/test_wakeword.py --featurizer <feat.onnx> --model <head.onnx> --audio sample.wav
```

### Where to go next

- **Deploy it** — both ONNX files together drop into the OVOS
  `ovos-ww-plugin-precise-onnx` plugin (see the quickstart's "Ship it" section for the
  `mycroft.conf` snippet).
- **Compare featurizers systematically** — `nb09_ablation.ipynb` sweeps featurizers ×
  losses × augmentation in one grid.
- **Train your own featurizer** — `nb10_audioset_featurizer.ipynb` builds a general-audio
  featurizer from scratch on AudioSet.

In [ ]:
import csv
import numpy as np
import torchaudio
from pathlib import Path
from ww_trainer.inference import OnnxWakeWordInferencer

# ── Inference test ────────────────────────────────────────────────────────────

head_p  = Path(_wh_result.get("head_onnx", "")) if _wh_result.get("head_onnx") else None
feat_p  = Path(FEATURIZER_ONNX)

if not feat_p.exists() or (head_p is None or not head_p.exists()):
    print("ONNX files not available — cannot run inference test.")
    print(f"  feat_onnx: {feat_p} ({'OK' if feat_p.exists() else 'MISSING'})")
    print(f"  head_onnx: {head_p} ({'OK' if (head_p and head_p.exists()) else 'MISSING'})")
else:
    inferencer = OnnxWakeWordInferencer(str(feat_p), str(head_p))

    _pos_path = None
    with open(test_csv) as f:
        for row in csv.reader(f):
            if len(row) >= 2 and row[1].strip() == "1" and Path(row[0]).exists():
                _pos_path = row[0]
                break

    if _pos_path:
        wav, sr = torchaudio.load(_pos_path)
        if sr != 16000:
            wav = torchaudio.functional.resample(wav, sr, 16000)
        score = inferencer.infer(wav.mean(0).numpy().astype(np.float32))
        print(f"Inference test: score={score:.4f}  ({'PASS' if score > 0.5 else 'LOW'})")
        print(f"  Sample: {Path(_pos_path).name}")
    else:
        print("No positive sample found in test CSV.")

print()
print("=" * 60)
print(f"WakeHuBERT classifier — {WAKE_WORD!r}")
print(f"  F1 = {_wh_result.get('f1', 0):.4f}")
if feat_p.exists() and head_p and head_p.exists():
    print(f"  .venv/bin/python scripts/eval/test_wakeword.py \\")
    print(f"      --featurizer {feat_p} \\")
    print(f"      --model      {head_p} \\")
    print(f"      --audio      sample.wav")
print("=" * 60)